In [ ]:
# Cluster task. Connects to Lakebase with a Databricks-minted OAuth token (no stored password).
import sys, os

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
nb_dir = os.path.dirname(ctx.notebookPath().get())
if ("/Workspace" + nb_dir) not in sys.path:
    sys.path.append("/Workspace" + nb_dir)
from state_store import StateStore


def widget(name, default=""):
    dbutils.widgets.text(name, default)
    return dbutils.widgets.get(name)


db_instance = widget("db_instance", "lakefed-ingest-mt")
db_name = widget("db_name", "lakefed_ingest_mt")
store = StateStore.connect_lakebase(db_instance, db_name)

In [ ]:
# Insert example cohort rows into the Lakebase control table.
# Prereqs per table: CT enabled, PK present, ALLOW_SNAPSHOT_ISOLATION ON,
# CHANGE_RETENTION > expected seed time; a UC Connection per instance.
rows = [
    # (job_name, task_collection, src_type, src_connection, src_database, src_catalog,
    #  src_schema, src_table, sink_catalog, sink_schema, sink_table, enable_iceberg_reads,
    #  primary_key, sink_cluster_cols, load_type, load_partitioned, select_list,
    #  use_remote_query, ct_enabled, source_instance_group, tier, priority, warehouse_id, task_enabled)
    ("lakefed_ingest_mt", "example_cohort", "sqlserver", "sqlserver_instance_1", "tenant_db_001", None,
     "dbo", "orders", "lakefed_ingest_mt", "tenant_db_001", "orders", False,
     "order_id", None, "ct", False, "order_id, customer_id, status, updated_at",
     True, True, "group_a", "gold", 1, None, True),
    ("lakefed_ingest_mt", "example_cohort", "sqlserver", "sqlserver_instance_2", "tenant_db_002", None,
     "dbo", "orders", "lakefed_ingest_mt", "tenant_db_002", "orders", False,
     "order_id", None, "ct", False, "order_id, customer_id, status, updated_at",
     True, True, "group_b", "gold", 1, None, True),
]
cols = ("job_name, task_collection, src_type, src_connection, src_database, src_catalog, "
        "src_schema, src_table, sink_catalog, sink_schema, sink_table, enable_iceberg_reads, "
        "primary_key, sink_cluster_cols, load_type, load_partitioned, select_list, "
        "use_remote_query, ct_enabled, source_instance_group, tier, priority, warehouse_id, task_enabled")
placeholders = ", ".join(["%s"] * 24)
with store.connection.cursor() as cur:
    for r in rows:
        cur.execute(f"insert into lakefed_ingest_mt.control ({cols}) values ({placeholders})", r)
print(f"inserted {len(rows)} control rows")